# Qwen3.5-0.8B Full Dataset Training on Colab T4

This notebook trains Qwen3.5-0.8B on full DialogSum dataset (12,460 samples) using LoRA.

**Experiments:**
- exp0_qwen: Baseline (no length control)
- exp1_qwen: Length-controllable summarization
- exp1_multi_qwen: Multi-task (summarization + topic generation)

**Expected runtime on T4:** ~45-90 minutes per experiment (with optimizations)

In [ ]:
# @title 1. Install dependencies (run once)
import subprocess
import sys

# Update transformers to latest version (required for Qwen3.5)
print("Updating transformers to latest version...")
!pip install -q --upgrade transformers

# Core dependencies
print("Installing core dependencies...")
!pip install -q datasets peft rouge-score tqdm accelerate

# Try flash-attn, fallback to SDPA if fails
print("\nAttempting to install flash-attn...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "flash-attn", "--no-build-isolation"])
    USE_FLASH_ATTN = True
    print("Flash Attention 2 installed successfully!")
except:
    USE_FLASH_ATTN = False
    print("Flash-attn installation failed (this is OK). Using SDPA fallback.")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import json
from pathlib import Path
from typing import Dict, List
import random
import numpy as np

# Print transformers version
from transformers import __version__ as transformers_version
print(f"\nTransformers version: {transformers_version}")

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"Random seed set to {SEED}")

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Flash Attention: {USE_FLASH_ATTN}")

In [ ]:
# @title 2. Dataset and preprocessing utilities

# Length bucket definitions
SHORT_MAX = 15
MEDIUM_MAX = 35

BUCKET_RANGES = {
    "SHORT": (5, 15),
    "MEDIUM": (16, 35),
    "LONG": (36, 999),
}

LENGTH_INSTRUCTIONS = {
    "SHORT": "Write a very brief one-sentence summary of the dialogue in 5 to 15 words.",
    "MEDIUM": "Write a short summary of the dialogue in 16 to 35 words.",
    "LONG": "Write a detailed summary of the dialogue in more than 35 words.",
}

def get_length_bucket(summary: str) -> str:
    word_count = len(summary.split())
    if word_count <= SHORT_MAX:
        return "SHORT"
    if word_count <= MEDIUM_MAX:
        return "MEDIUM"
    return "LONG"

def preprocess_batch(batch, tokenizer, use_length_tokens=False, multitask=False):
    """Batch preprocessing for causal LM."""
    all_input_ids = []
    all_attention_mask = []
    all_labels = []
    
    for dialogue, summary, topic in zip(batch['dialogue'], batch['summary'], batch['topic']):
        pairs = []
        
        # Summarize task
        if use_length_tokens:
            bucket = get_length_bucket(summary)
            length_instr = LENGTH_INSTRUCTIONS[bucket]
            prompt = (
                f"Summarize the following dialogue.\\n"
                f"Instruction: {length_instr}\\n"
                f"Dialogue:\\n{dialogue}\\n"
                f"Summary: "
            )
        else:
            prompt = (
                f"Summarize the following dialogue.\\n"
                f"Dialogue:\\n{dialogue}\\n"
                f"Summary: "
            )
        pairs.append((prompt, summary))
        
        # Topic task (multi-task only)
        if multitask:
            topic_prompt = (
                f"What is the topic of the following dialogue? Answer in a short phrase.\\n"
                f"Dialogue:\\n{dialogue}\\n"
                f"Topic: "
            )
            pairs.append((topic_prompt, topic))
        
        for prompt_text, target_text in pairs:
            prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
            target_ids = tokenizer.encode(
                target_text + tokenizer.eos_token, add_special_tokens=False
            )
            
            # Truncate prompt if needed (keep target)
            max_total = 256 + 64
            if len(prompt_ids) + len(target_ids) > max_total:
                prompt_ids = prompt_ids[-(max_total - len(target_ids)):]
            
            full_ids = prompt_ids + target_ids
            labels = [-100] * len(prompt_ids) + target_ids
            attn_mask = [1] * len(full_ids)
            
            all_input_ids.append(full_ids)
            all_attention_mask.append(attn_mask)
            all_labels.append(labels)
    
    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels,
    }

# Load dataset
print("Loading DialogSum dataset...")
ds = load_dataset("knkarthick/dialogsum")
print(f"Train: {len(ds['train'])}, Val: {len(ds['validation'])}, Test: {len(ds['test'])}")

In [ ]:
# @title 3. Model loading utilitiesdef load_qwen_model(tokenizer, use_flash_attn):    """Load Qwen3.5-0.8B with LoRA."""    # Choose attention implementation    attn_impl = "flash_attention_2" if use_flash_attn else "sdpa"    print(f"Using attention implementation: {attn_impl}")        model = AutoModelForCausalLM.from_pretrained(        "Qwen/Qwen3.5-0.8B",        dtype=torch.bfloat16,        attn_implementation=attn_impl,    )        # LoRA config - expanded target modules for better adaptation    lora_cfg = LoraConfig(        r=16,        lora_alpha=32,        target_modules=["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],        lora_dropout=0.05,        bias="none",        task_type=TaskType.CAUSAL_LM,    )    model = get_peft_model(model, lora_cfg)    model.print_trainable_parameters()        # Enable gradient checkpointing for memory efficiency    model.gradient_checkpointing_enable()    model.enable_input_require_grads()    model.config.use_cache = False        return model# Initialize tokenizertokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B", padding_side="left")if tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_tokenprint(f"Tokenizer: {len(tokenizer)} tokens")

In [ ]:
# @title 4. Training function

def train_experiment(exp_name, use_length_tokens, multitask, epochs=5, batch_size=16):
    """Train a single experiment with optimizations."""
    print(f"\n{'='*60}")
    print(f"Training: {exp_name}")
    print(f"Length tokens: {use_length_tokens}, Multi-task: {multitask}")
    print(f"Batch size: {batch_size}, Epochs: {epochs}")
    print(f"{'='*60}")
    
    # Preprocess with map() for speed
    print("Preprocessing data...")
    train_tok = ds['train'].map(
        preprocess_batch,
        fn_kwargs={"tokenizer": tokenizer, "use_length_tokens": use_length_tokens, "multitask": multitask},
        batched=True,
        batch_size=1000,
        remove_columns=ds['train'].column_names,
        desc="Processing train",
    )
    val_tok = ds['validation'].map(
        preprocess_batch,
        fn_kwargs={"tokenizer": tokenizer, "use_length_tokens": use_length_tokens, "multitask": multitask},
        batched=True,
        batch_size=1000,
        remove_columns=ds['validation'].column_names,
        desc="Processing val",
    )
    print(f"Train samples: {len(train_tok)}, Val samples: {len(val_tok)}")
    
    # Load model
    model = load_qwen_model(tokenizer, USE_FLASH_ATTN)
    model = model.to(device)
    
    # Output dir
    output_dir = Path(f"/content/drive/MyDrive/qwen_colab/{exp_name}")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Training arguments with optimizations
    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=1,
        learning_rate=5e-5,
        warmup_steps=100,
        weight_decay=0.01,
        max_grad_norm=1.0,  # Gradient clipping
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=500,  # Evaluate every 500 steps
        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        bf16=True,
        fp16=False,
        dataloader_num_workers=2,  # Reduce workers for stability
        dataloader_pin_memory=True,  # Faster data loading
        report_to="none",
        seed=SEED,  # Ensure reproducibility
    )
    
    # Data collator (right-pad)
    class CausalLMDataCollator:
        def __init__(self, pad_token_id):
            self.pad_token_id = pad_token_id
        def __call__(self, features: List[Dict]) -> Dict:
            max_len = max(len(f["input_ids"]) for f in features)
            batch = {"input_ids": [], "attention_mask": [], "labels": []}
            for f in features:
                pad_len = max_len - len(f["input_ids"])
                batch["input_ids"].append(f["input_ids"] + [self.pad_token_id] * pad_len)
                batch["attention_mask"].append(f["attention_mask"] + [0] * pad_len)
                batch["labels"].append(f["labels"] + [-100] * pad_len)
            return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}
    
    data_collator = CausalLMDataCollator(pad_token_id=tokenizer.pad_token_id)
    
    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    
    # Train
    print("\nStarting training...")
    train_result = trainer.train()
    trainer.save_model()
    
    # Save complete state (model + tokenizer + config)
    tokenizer.save_pretrained(output_dir)
    
    metrics = train_result.metrics
    config_data = {
        "experiment": exp_name,
        "model_name": "Qwen/Qwen3.5-0.8B",
        "lora_r": 16,
        "lora_alpha": 32,
        "use_length_tokens": use_length_tokens,
        "multitask": multitask,
        "train_samples": len(train_tok),
        "val_samples": len(val_tok),
        "epochs": epochs,
        "batch_size": batch_size,
        "learning_rate": 5e-5,
        "flash_attn": USE_FLASH_ATTN,
        "seed": SEED,
        "metrics": metrics,
    }
    
    metrics_path = output_dir / "training_metrics.json"
    config_path = output_dir / "config.json"
    
    with open(metrics_path, 'w') as f:
        json.dump(config_data, f, indent=2)
    with open(config_path, 'w') as f:
        json.dump({k: v for k, v in config_data.items() if k != "metrics"}, f, indent=2)
    
    print(f"\nTraining complete!")
    print(f"Saved to: {output_dir}")
    print(f"Final train loss: {metrics.get('train_loss', 'N/A'):.4f}")
    
    return trainer, metrics

In [ ]:
# @title 5. Mount Google Drive (for saving checkpoints)
from google.colab import drive
drive.mount('/content/drive')

## Run Experiments

Run each cell below to train a different experiment. You can run them sequentially or one at a time.

**Note:** Batch size 16 uses more VRAM. If OOM, reduce to 8.

In [ ]:
# @title Exp 0: Baseline (no length control)
trainer_exp0, metrics_exp0 = train_experiment(
    exp_name="exp0_qwen_full",
    use_length_tokens=False,
    multitask=False,
    epochs=5,
    batch_size=16
)

In [ ]:
# @title Exp 1: Length-controllable summarization
trainer_exp1, metrics_exp1 = train_experiment(
    exp_name="exp1_qwen_full",
    use_length_tokens=True,
    multitask=False,
    epochs=5,
    batch_size=16
)

In [ ]:
# @title Exp 2: Multi-task (summarization + topic)
trainer_exp2, metrics_exp2 = train_experiment(
    exp_name="exp1_multi_qwen_full",
    use_length_tokens=True,
    multitask=True,
    epochs=5,
    batch_size=16
)

## Evaluation (run in Colab)

Evaluate ROUGE metrics directly in Colab without downloading models locally.

In [ ]:
# @title Evaluate model (run after training)from tqdm import tqdmfrom rouge_score import rouge_scorerdef evaluate_in_colab(exp_name, max_samples=500):    """Evaluate a trained model on test set."""    model_dir = Path(f"/content/drive/MyDrive/qwen_colab/{exp_name}")    if not model_dir.exists():        print(f"Error: {model_dir} not found. Train the model first.")        return        # Load config to get training params    with open(model_dir / "config.json", 'r') as f:        config_data = json.load(f)        print(f"Evaluating: {exp_name}")    print(f"Model: {config_data['model_name']}")    print(f"Use length tokens: {config_data['use_length_tokens']}")        # Load model and tokenizer    attn_impl = "flash_attention_2" if USE_FLASH_ATTN else "sdpa"    from peft import PeftConfig, PeftModel        peft_config = PeftConfig.from_pretrained(str(model_dir))    base_model = AutoModelForCausalLM.from_pretrained(        peft_config.base_model_name_or_path,        dtype=torch.bfloat16,        attn_implementation=attn_impl,    )    base_model.resize_token_embeddings(len(tokenizer))    model = PeftModel.from_pretrained(base_model, str(model_dir))    model = model.to(device).eval()        # Load test data    test_data = ds['test'].select(range(min(max_samples, len(ds['test']))))    use_len = config_data['use_length_tokens']        # Build prompts    prompts = []    ref_summaries = []    target_buckets = []        for row in test_data:        ref_summaries.append(row['summary'])        target_buckets.append(get_length_bucket(row['summary']))                if use_len:            bucket = get_length_bucket(row['summary'])            length_instr = LENGTH_INSTRUCTIONS[bucket]            prompt = (                f"Summarize the following dialogue.\\n"                f"Instruction: {length_instr}\\n"                f"Dialogue:\\n{row['dialogue']}\\n"                f"Summary: "            )        else:            prompt = (                f"Summarize the following dialogue.\\n"                f"Dialogue:\\n{row['dialogue']}\\n"                f"Summary: "            )        prompts.append(prompt)        # Generate summaries    print(f"\nGenerating summaries for {len(prompts)} samples...")    predictions = []    batch_size = 8        for i in tqdm(range(0, len(prompts), batch_size)):        batch = prompts[i:i+batch_size]        tokenized = tokenizer(            batch,            max_length=256,            truncation=True,            padding=True,            return_tensors="pt",        ).to(device)                prompt_len = tokenized["input_ids"].shape[1]                with torch.no_grad():            output_ids = model.generate(                **tokenized,                max_new_tokens=64,                pad_token_id=tokenizer.pad_token_id,                eos_token_id=tokenizer.eos_token_id,                do_sample=False,                num_beams=1,            )                new_tokens = output_ids[:, prompt_len:]        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)        predictions.extend([d.strip() for d in decoded])        # Compute ROUGE    print("\nComputing ROUGE scores...")    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)    rouge1_scores = []    rouge2_scores = []    rougeL_scores = []        for pred, ref in zip(predictions, ref_summaries):        scores = scorer.score(ref, pred)        rouge1_scores.append(scores['rouge1'].fmeasure)        rouge2_scores.append(scores['rouge2'].fmeasure)        rougeL_scores.append(scores['rougeL'].fmeasure)        results = {        "experiment": exp_name,        "num_samples": len(predictions),        "rouge1": round(sum(rouge1_scores) / len(rouge1_scores) * 100, 2),        "rouge2": round(sum(rouge2_scores) / len(rouge2_scores) * 100, 2),        "rougeL": round(sum(rougeL_scores) / len(rougeL_scores) * 100, 2),    }        # Length accuracy (if applicable)    if use_len:        correct = 0        for pred, bucket in zip(predictions, target_buckets):            wc = len(pred.split())            lo, hi = BUCKET_RANGES[bucket]            if lo <= wc <= hi:                correct += 1        results["length_accuracy"] = round(correct / len(predictions), 4)                # Per-bucket accuracy        for b in ["SHORT", "MEDIUM", "LONG"]:            idxs = [i for i, bk in enumerate(target_buckets) if bk == b]            if idxs:                b_preds = [predictions[i] for i in idxs]                b_targets = [target_buckets[i] for i in idxs]                b_correct = sum(                    1 for p, t in zip(b_preds, b_targets)                    if BUCKET_RANGES[t][0] <= len(p.split()) <= BUCKET_RANGES[t][1]                )                results[f"length_acc_{b.lower()}"] = round(b_correct / len(b_preds), 4)        # Save results    RESULTS_DIR = model_dir    results_path = RESULTS_DIR / "eval_results.json"    with open(results_path, 'w') as f:        json.dump(results, f, indent=2)        # Print summary    print(f"\n{'='*60}")    print(f"Results: {exp_name}")    print(f"{'='*60}")    print(f"ROUGE-1:  {results['rouge1']:.2f}")    print(f"ROUGE-2:  {results['rouge2']:.2f}")    print(f"ROUGE-L:  {results['rougeL']:.2f}")        if 'length_accuracy' in results:        print(f"Length Accuracy: {results['length_accuracy']*100:.1f}%")        print(f"  SHORT: {results.get('length_acc_short', 0)*100:.1f}%")        print(f"  MEDIUM: {results.get('length_acc_medium', 0)*100:.1f}%")        print(f"  LONG: {results.get('length_acc_long', 0)*100:.1f}%")    print(f"\nSaved to: {results_path}")        return results

In [ ]:
# @title Evaluate all trained models
# Run this after all three experiments complete
for exp in ["exp0_qwen_full", "exp1_qwen_full", "exp1_multi_qwen_full"]:
    evaluate_in_colab(exp, max_samples=500)

## Download Checkpoints

After training and evaluation, download models to your local machine:

In [ ]:
# @title Download all checkpoints as ZIP
!cd /content/drive/MyDrive/qwen_colab && zip -r qwen_models.zip exp0_qwen_full exp1_qwen_full exp1_multi_qwen_full

from google.colab import files
files.download('/content/drive/MyDrive/qwen_colab/qwen_models.zip')

In [ ]:
# @title Download individual checkpoint
# Change 'exp0_qwen_full' to experiment you want
!cd /content/drive/MyDrive/qwen_colab/exp0_qwen_full && zip -r exp0_qwen_full.zip .
files.download('/content/drive/MyDrive/qwen_colab/exp0_qwen_full/exp0_qwen_full.zip')